# 01 — Dataset understanding

**Delivery ETA Intelligence**

This notebook is your first look at the delivery dataset. We load the raw file, inspect structure and quality, and plan deeper analysis.

## Project objective

**Delivery ETA Intelligence** combines machine learning and graph analytics to:

- **Predict** how long deliveries will take (estimated time of arrival, ETA) at the segment or trip level.
- **Map** logistics as a network of centers (nodes) and routes (edges).
- **Find bottlenecks** — hubs or lanes where delays cluster — using graph metrics and operational data.

The end goal is more accurate ETAs for customers and better planning for operations teams.

## Dataset overview

We use **`delivery_data.csv`** in `data/raw/`. Each row typically describes a **route segment** within a trip:

- **Trip & route IDs** — e.g. `trip_uuid`, `route_schedule_uuid`, `route_type`
- **Locations** — `source_center`, `destination_center`, and human-readable names
- **Timestamps** — trip creation, segment start/end (`od_start_time`, `od_end_time`)
- **Time & distance** — `actual_time`, `osrm_time`, distances, and **factors** comparing actual vs. routing-engine estimates
- **Cutoff flags** — operational windows (`is_cutoff`, `cutoff_factor`)

The column `data` indicates train vs. test split labels where applicable.  
Understanding these fields is the foundation for feature engineering and graph construction later.

## Setup — imports and paths

Run this cell first. The data path is relative to the **project root** (parent of `notebooks/`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project root = folder that contains data/, src/, notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "delivery_data.csv"
print(f"Loading: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Loading: D:\delivery eta\data\raw\delivery_data.csv
File exists: True


In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")

Loaded 144,867 rows and 24 columns.


## First look — sample rows

In [3]:
df.head()

,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,cutoff_timestamp,actual_distance_to_destination,actual_time,osrm_time,osrm_distance,factor,segment_actual_time,segment_osrm_time,segment_osrm_distance,segment_factor
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:27:55,10.435660,14.0,11.0,11.9653,1.272727,14.0,11.0,11.9653,1.272727
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:17:55,18.936842,24.0,20.0,21.7243,1.200000,10.0,9.0,9.7590,1.111111
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 04:01:19.505586,27.637279,40.0,28.0,32.5395,1.428571,16.0,7.0,10.8152,2.285714
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:39:57,36.118028,62.0,40.0,45.5620,1.550000,21.0,12.0,13.0224,1.750000
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,2018-09-20 03:33:55,39.386040,68.0,44.0,54.2181,1.545455,6.0,5.0,3.9153,1.200000


## Shape (rows × columns)

In [4]:
print("Shape:", df.shape)
print(f"  Rows:    {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

Shape: (144867, 24)
  Rows:    144,867
  Columns: 24


## Column names

In [5]:
print("Column names:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {col}")

Column names:
   1. data
   2. trip_creation_time
   3. route_schedule_uuid
   4. route_type
   5. trip_uuid
   6. source_center
   7. source_name
   8. destination_center
   9. destination_name
  10. od_start_time
  11. od_end_time
  12. start_scan_to_end_scan
  13. is_cutoff
  14. cutoff_factor
  15. cutoff_timestamp
  16. actual_distance_to_destination
  17. actual_time
  18. osrm_time
  19. osrm_distance
  20. factor
  21. segment_actual_time
  22. segment_osrm_time
  23. segment_osrm_distance
  24. segment_factor


## Data types and non-null counts (`info`)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 144867 entries, 0 to 144866
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   data                            144867 non-null  str    
 1   trip_creation_time              144867 non-null  str    
 2   route_schedule_uuid             144867 non-null  str    
 3   route_type                      144867 non-null  str    
 4   trip_uuid                       144867 non-null  str    
 5   source_center                   144867 non-null  str    
 6   source_name                     144574 non-null  str    
 7   destination_center              144867 non-null  str    
 8   destination_name                144606 non-null  str    
 9   od_start_time                   144867 non-null  str    
 10  od_end_time                     144867 non-null  str    
 11  start_scan_to_end_scan          144867 non-null  float64
 12  is_cutoff                  

## Numeric summary (`describe`)

In [7]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
data,144867,2,training,104858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_creation_time,144867,14817,2018-09-28 05:23:15.359220,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_schedule_uuid,144867,1504,thanos::sroute:4029a8a2-6c74-4b7e-a6d8-f9e069f...,1812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_type,144867,2,FTL,99660,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trip_uuid,144867,14817,trip-153811219535896559,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_center,144867,1508,IND000000ACB,23347,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_name,144574,1498,Gurgaon_Bilaspur_HB (Haryana),23347,NaN,NaN,NaN,NaN,NaN,NaN,NaN
destination_center,144867,1481,IND000000ACB,15192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
destination_name,144606,1468,Gurgaon_Bilaspur_HB (Haryana),15192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
od_start_time,144867,26369,2018-09-21 18:37:09.322207,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing values summary

Columns with missing data may need imputation, dropping, or special handling in preprocessing.

In [8]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "column": missing.index,
    "missing_count": missing.values,
    "missing_pct": missing_pct.values,
}).sort_values("missing_count", ascending=False)

print("Columns with any missing values:")
display(missing_summary[missing_summary["missing_count"] > 0])

if missing_summary["missing_count"].sum() == 0:
    print("\nNo missing values detected.")
else:
    print(f"\nTotal missing cells: {missing_summary['missing_count'].sum():,}")

Columns with any missing values:


,column,missing_count,missing_pct
6,source_name,293,0.20
8,destination_name,261,0.18



Total missing cells: 554
